CLIP官方代码：https://github.com/openai/CLIP/blob/main/clip/model.py

<div id='top'>包括以下操作：</div>
<li><a href='#1'>111</a></li>
<li><a href='#2'></a></li>
<li><a href='#3'>class VisionTransformer</a></li>
<li><a href='#4'></a></li>
<li><a href='#5'></a></li>
<li><a href='#6'></a></li>

In [ ]:
from collections import OrderedDict
from typing import Tuple, Union

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

尽管文档中没有明确提到 requires_grad 的设置，但可以放心地使用 nn.Linear 层进行训练，因为它的参数会自动设置为可训练状态。

In [1]:
import torch
import torch.nn as nn

# 创建一个 Linear 层
linear_layer = nn.Linear(in_features=10, out_features=5)

# 检查权重和偏置的 requires_grad 属性
print(linear_layer.weight.requires_grad)  # 输出: True
print(linear_layer.bias.requires_grad)    # 输出: True

True
True


In [2]:
'''
    在PyTorch中定义如AttentionPool2d的自定义模块时，无需显式定义反向传播函数，原因在于PyTorch的自动微分（Autograd）机制会自动构建计算图并推导梯度。
    在PyTorch中，nn.Module的子模块（如nn.Linear）和通过nn.Parameter定义的参数默认启用梯度追踪，无需显式设置requires_grad=True。
'''
class AttentionPool2d(nn.Module):
    def __init__(self, spacial_dim: int, embed_dim: int, num_heads: int, output_dim: int = None):
        super().__init__()
        # nn.Parameter（位置编码positional_embedding）：作为可学习参数，其梯度由Autograd自动追踪。
        self.positional_embedding = nn.Parameter(torch.randn(spacial_dim ** 2 + 1, embed_dim) / embed_dim ** 0.5)
        '''
            nn.Linear层（q_proj, k_proj, v_proj, c_proj）：前向计算：y = xW^T + b；梯度计算：PyTorch自动计算对W和b的梯度，无需手动实现。
            nn.Linear层内部权重和偏置均为nn.Parameter，默认requires_grad=True
            https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear
            nn.linear Applies an affine linear transformation to the incoming data: y=xA^T+b.
        '''
        self.k_proj = nn.Linear(embed_dim, embed_dim) # key projection
        self.q_proj = nn.Linear(embed_dim, embed_dim) # query projection
        self.v_proj = nn.Linear(embed_dim, embed_dim) # value projection

        # 下面这行代码是CLIP跨模态对齐的核心：将视觉特征投影到与文本特征相同的语义空间（如512维），使图像和文本嵌入可直接比较。
        self.c_proj = nn.Linear(embed_dim, output_dim or embed_dim) # 最终输出投影层（跨模态对齐）
        self.num_heads = num_heads

    def forward(self, x):
        x = x.flatten(start_dim=2).permute(2, 0, 1)  # NCHW -> (HW)NC
        x = torch.cat([x.mean(dim=0, keepdim=True), x], dim=0)  # (HW+1)NC
        x = x + self.positional_embedding[:, None, :].to(x.dtype)  # (HW+1)NC
        x, _ = F.multi_head_attention_forward(
            query=x[:1], key=x, value=x,
            embed_dim_to_check=x.shape[-1],
            num_heads=self.num_heads,
            q_proj_weight=self.q_proj.weight,
            k_proj_weight=self.k_proj.weight,
            v_proj_weight=self.v_proj.weight,
            in_proj_weight=None,
            in_proj_bias=torch.cat([self.q_proj.bias, self.k_proj.bias, self.v_proj.bias]),
            bias_k=None,
            bias_v=None,
            add_zero_attn=False,
            dropout_p=0,
            out_proj_weight=self.c_proj.weight,
            out_proj_bias=self.c_proj.bias,
            use_separate_proj_weight=True,
            training=self.training,
            need_weights=False
        )
        return x.squeeze(0)
    
    '''
        PyTorch的Autograd系统通过动态构建计算图、链式法则梯度传播及灵活的内存管理，实现了对任意张量操作的自动微分。其核心体现在：
        张量属性（requires_grad、grad_fn、grad）明确标记梯度状态与计算历史。
    '''

通过下段代码示例验证AttentionPool2d参数的默认梯度行为：

In [3]:
# 初始化模块
attnpool = AttentionPool2d(
    spacial_dim=7,  # 假设输入分辨率224，经过下采样32倍后为7x7
    embed_dim=512,
    num_heads=8,
    output_dim=512
)

# 检查位置编码的requires_grad
print(attnpool.positional_embedding.requires_grad)  # 输出: True  # nn.Parameter，默认requires_grad=True

# 检查k_proj权重的requires_grad
print(attnpool.k_proj.weight.requires_grad)        # 输出: True  # nn.Linear层内部权重和偏置均为nn.Parameter，默认requires_grad=True


True
True


原版ResNet（如torchvision实现）在输入阶段使用单层7x7卷积作为初始特征提取（stem），后接最大池化层。</br>
而ModifiedResNet将其改为三层3x3卷积组成的stem，并采用平均池化替代最大池化。</br>
这种设计源于ResNet-D的改进：</br>
1、三层小卷积的优势：多个3x3卷积堆叠可达到与7x7卷积相同的感受野，同时减少参数量并增强非线性表达能力。</br>
2、平均池化的作用：相比最大池化，平均池化能保留更多整体特征信息，减少高频细节丢失，更适合多模态任务中对全局语义的捕捉。

In [ ]:
class ModifiedResNet(nn.Module):
    """
    A ResNet class that is similar to torchvision's but contains the following changes:
    - There are now 3 "stem" convolutions as opposed to 1, with an average pool instead of a max pool.
    - Performs anti-aliasing strided convolutions, where an avgpool is prepended to convolutions with stride > 1
    - The final pooling layer is a QKV attention instead of an average pool
    """

    def __init__(self, layers, output_dim, heads, input_resolution=224, width=64):
        super().__init__()
        self.output_dim = output_dim
        self.input_resolution = input_resolution

        # the 3-layer stem
        '''
        原版ResNet（如torchvision实现）在输入阶段使用单层7x7卷积作为初始特征提取（stem），后接最大池化层。
        而ModifiedResNet将其改为三层3x3卷积组成的stem，并采用平均池化替代最大池化。
        这种设计源于ResNet-D的改进：
        1、三层小卷积的优势：多个3x3卷积堆叠可达到与7x7卷积相同的感受野，同时减少参数量并增强非线性表达能力。
        2、平均池化的作用：相比最大池化，平均池化能保留更多整体特征信息，减少高频细节丢失，更适合多模态任务中对全局语义的捕捉。
        '''
        self.conv1 = nn.Conv2d(3, width // 2, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width // 2)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(width // 2, width // 2, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(width // 2)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv3 = nn.Conv2d(width // 2, width, kernel_size=3, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(width)
        self.relu3 = nn.ReLU(inplace=True)
        self.avgpool = nn.AvgPool2d(2)

        # residual layers
        self._inplanes = width  # this is a *mutable* variable used during construction
        self.layer1 = self._make_layer(width, layers[0])
        self.layer2 = self._make_layer(width * 2, layers[1], stride=2)
        self.layer3 = self._make_layer(width * 4, layers[2], stride=2)
        self.layer4 = self._make_layer(width * 8, layers[3], stride=2)

        embed_dim = width * 32  # the ResNet feature dimension
        self.attnpool = AttentionPool2d(input_resolution // 32, embed_dim, heads, output_dim) # 通过这个实现投影层模态对齐：AttentionPool2d 的 c_proj

    def _make_layer(self, planes, blocks, stride=1):
        layers = [Bottleneck(self._inplanes, planes, stride)]

        self._inplanes = planes * Bottleneck.expansion
        for _ in range(1, blocks):
            layers.append(Bottleneck(self._inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        def stem(x):
            x = self.relu1(self.bn1(self.conv1(x)))
            x = self.relu2(self.bn2(self.conv2(x)))
            x = self.relu3(self.bn3(self.conv3(x)))
            x = self.avgpool(x)
            return x

        x = x.type(self.conv1.weight.dtype)
        x = stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.attnpool(x)

        return x

![1.png](1.png)

<div style='color:skyblue; font-size:24px' id='3'>class VisionTransformer</div>

<a href='#top'>▲ Top</a>

In [ ]:
class VisionTransformer(nn.Module):
    '''
        input_resolution: 输入图像分辨率（如224x224）。
        patch_size: 图像分块大小（如16x16）。
        width: Transformer的嵌入维度（即d_model，如768, 16*16*3 = 768）。
        layers: Transformer编码器层数。
        heads: 多头注意力头数。
        output_dim: 最终输出的特征维度（如CLIP的文本-图像对齐维度）。
    '''
    def __init__(self, input_resolution: int, patch_size: int, width: int, layers: int, heads: int, output_dim: int):
        super().__init__()
        self.input_resolution = input_resolution
        self.output_dim = output_dim
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=width, kernel_size=patch_size, stride=patch_size, bias=False)

        scale = width ** -0.5
        self.class_embedding = nn.Parameter(scale * torch.randn(width))
        self.positional_embedding = nn.Parameter(scale * torch.randn((input_resolution // patch_size) ** 2 + 1, width))
        self.ln_pre = LayerNorm(width)

        self.transformer = Transformer(width, layers, heads)

        self.ln_post = LayerNorm(width)
        # 投影层：
        self.proj = nn.Parameter(scale * torch.randn(width, output_dim))

    def forward(self, x: torch.Tensor):
        x = self.conv1(x)  # shape = [*, width, grid, grid]
        x = x.reshape(x.shape[0], x.shape[1], -1)  # shape = [*, width, grid ** 2]
        x = x.permute(0, 2, 1)  # shape = [*, grid ** 2, width]
        x = torch.cat([self.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device), x], dim=1)  # shape = [*, grid ** 2 + 1, width]
        x = x + self.positional_embedding.to(x.dtype)
        x = self.ln_pre(x)

        x = x.permute(1, 0, 2)  # NLD -> LND # 调整维度为 [序列长度, 批大小, 维度]
        x = self.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD # 恢复为 [批大小, 序列长度, 维度]

        x = self.ln_post(x[:, 0, :])

        if self.proj is not None:
            x = x @ self.proj

        return x

In [ ]:
class CLIP(nn.Module):
    '''
        embed_dim: 图像和文本的嵌入维度，最终对齐的特征向量维度（例如512）。
        image_resolution: 输入图像的分辨率（例如224x224）。
        vision_layers: 视觉模型的层配置。若为元组（如ResNet的层数），则使用ModifiedResNet；若为整数（如ViT的层数），则使用VisionTransformer。
        vision_width: 视觉模型的特征维度（如ResNet的通道数或ViT的隐藏层维度）。
        vision_patch_size: ViT中将图像分割的块大小（如16x16）。
        context_length: 文本输入的最大长度（如77个token）。
        vocab_size: 文本词汇表的大小（如21128个词）。
        transformer_width: 文本Transformer的隐藏层维度（如512）。
        transformer_heads: 文本Transformer的多头注意力头数（如8头）。
        transformer_layers: 文本Transformer的层数（如12层）。
    '''
    def __init__(self,
                 embed_dim: int,
                 # vision
                 image_resolution: int,
                 vision_layers: Union[Tuple[int, int, int, int], int],
                 vision_width: int,
                 vision_patch_size: int,
                 # text
                 context_length: int,
                 vocab_size: int,
                 transformer_width: int,
                 transformer_heads: int,
                 transformer_layers: int
                 ):
        super().__init__()

        self.context_length = context_length

        if isinstance(vision_layers, (tuple, list)):
            vision_heads = vision_width * 32 // 64
            self.visual = ModifiedResNet(
                layers=vision_layers,
                output_dim=embed_dim,
                heads=vision_heads,
                input_resolution=image_resolution,
                width=vision_width
            )
        else:
            vision_heads = vision_width // 64
            self.visual = VisionTransformer(
                input_resolution=image_resolution,
                patch_size=vision_patch_size,
                width=vision_width,
                layers=vision_layers,
                heads=vision_heads,
                output_dim=embed_dim
            )

        self.transformer = Transformer(
            width=transformer_width,
            layers=transformer_layers,
            heads=transformer_heads,
            attn_mask=self.build_attention_mask()
        )

        self.vocab_size = vocab_size
        self.token_embedding = nn.Embedding(vocab_size, transformer_width)
        self.positional_embedding = nn.Parameter(torch.empty(self.context_length, transformer_width))
        self.ln_final = LayerNorm(transformer_width)

        # 将归一化后的文本特征投影到embed_dim，与图像特征对齐
        self.text_projection = nn.Parameter(torch.empty(transformer_width, embed_dim))
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

        self.initialize_parameters()